In [ ]:
"""LangGraph A2A conversational agent.

Supports the A2A protocol with messages input for conversational interactions.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Any, Dict, List, TypedDict

from langgraph.graph import StateGraph
from langgraph.runtime import Runtime
from openai import AsyncOpenAI
from langgraph_sdk import get_client

class Context(TypedDict):
    """Context parameters for the agent."""
    my_configurable_param: str


@dataclass
class State:
    """Input state for the agent.

    Defines the initial structure for A2A conversational messages.
    """
    messages: List[Dict[str, Any]]


async def call_model(state: State, runtime: Runtime[Context]) -> Dict[str, Any]:
    """Process conversational messages and returns output using OpenAI."""
    # Initialize OpenAI client
    #client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))



    client = get_client(url="http://127.0.0.1:2024")

    assistant = await client.assistants.create(
        "agent",
        context={},
        name="My Agent"
    )
    print(assistant)

# {'assistant_id': 'fef24173-0159-4297-9ee4-64018608e55f', 'graph_id': 'agent', 'config': {}, 'context': {}, 'metadata': {}, 'name': 'My Agent', 'created_at': '2026-06-14T01:55:11.098045+00:00', 'updated_at': '2026-06-14T01:55:11.098045+00:00', 'version': 1, 'description': None}
# Nice to meet you! I'm an AI conversationalist designed to provide helpful and informative responses.

    client = AsyncOpenAI(
            api_key="ollama",  # anything
            base_url="http://localhost:11434/v1"
        )

    # Process the incoming messages
    latest_message = state.messages[-1] if state.messages else {}
    user_content = latest_message.get("content", "No message content")

    # Create messages for OpenAI API
    openai_messages = [
        {
            "role": "system",
            "content": "You are a helpful conversational agent. Keep responses brief and engaging."
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    try:
        # Make OpenAI API call
        response = await client.chat.completions.create(
            #model="gpt-3.5-turbo",
            model="llama3.2",
            messages=openai_messages,
            max_tokens=100,
            temperature=0.7
        )
        

        # assistants = await client.assistants.list()
        # for a in assistants:
        #     print(a["assistant_id"], a["graph_id"])



        ai_response = response.choices[0].message.content

    except Exception as e:
        ai_response = f"I received your message but had trouble processing it. Error: {str(e)[:500]}..."

    # Create a response message
    response_message = {
        "role": "assistant",
        "content": ai_response
    }

    return {
        "messages": state.messages + [response_message]
    }


# Define the graph
graph = (
    StateGraph(State, context_schema=Context)
    .add_node(call_model)
    .add_edge("__start__", "call_model")
    .compile()
)




result = await graph.ainvoke({
    "messages": [
        {"role": "user", "content": "hi there. pls introduce yourself, your model id and when you are trained up to?"}
    ]
})

print(result["messages"][-1]["content"])




{'assistant_id': 'fef24173-0159-4297-9ee4-64018608e55f', 'graph_id': 'agent', 'config': {}, 'context': {}, 'metadata': {}, 'name': 'My Agent', 'created_at': '2026-06-14T01:55:11.098045+00:00', 'updated_at': '2026-06-14T01:55:11.098045+00:00', 'version': 1, 'description': None}
Nice to meet you! I'm an AI conversationalist designed to provide helpful and informative responses.

My official name is LLaMA, which stands for "Large Language Model Application." My model ID is `LLaMA-2`, indicating that I'm a version 2 of the LLaMA model.

I was trained up to December 2023, so my knowledge cutoff is around then. However, I can still provide general information and answer questions based on my training data. If you
